In [1]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Spin glass Hamiltonian ---
def spin_glass_energy(params, J):
    spins = torch.tanh(params)
    E = -torch.sum(J * torch.outer(spins, spins))
    return E / 2.0

# ----------------------------------------------------------------------
# NDDA computation (pure indicator, no update)
# ----------------------------------------------------------------------
def compute_NDDA(params, J, alpha=1.3):
    params = params.clone().detach().to(device).requires_grad_(True)
    d = params.numel()

    epsilon = torch.randn_like(params)
    params_1 = params + epsilon
    E1 = spin_glass_energy(params_1, J)
    E1.backward()
    I1 = epsilon * params.grad
    params.grad.zero_()

    mask_descent = (I1 < 0).to(device)
    mask_ascent  = (I1 > 0).to(device)
    epsilon_prime = torch.where(mask_descent, alpha * epsilon, epsilon)
    epsilon_prime = torch.where(mask_ascent, -epsilon, epsilon_prime)

    params_2 = params + epsilon_prime
    E2 = spin_glass_energy(params_2, J)
    E2.backward()
    I2 = epsilon_prime * params.grad
    params.grad.zero_()

    ndda_count = torch.sum((I1 >= 0) & (I2 >= 0)).float()
    return (ndda_count / d).item()

# ----------------------------------------------------------------------
# CPA optimizer with Hessian analysis
# ----------------------------------------------------------------------
def optimize_CPA(params_init, J, sigma_begin=0.02, sigma_end=0.001, epochs=200, alpha=1.3):
    J = J.clone().detach().to(device).requires_grad_(False)
    params = params_init.clone().detach().to(device).requires_grad_(True)

    losses = []
    ndda_vals = []
    min_eigenvalues = []       # most negative eigenvalue each 100 epochs
    len_of_params = params.numel()

    for epoch in range(epochs):
        sigma = sigma_begin * (1. - epoch/epochs) + sigma_end * epoch/epochs


        # --- CPA two-probe ---
        epsilon = (sigma * torch.randn_like(params)).detach().to(device).requires_grad_(False)
        params_1 = params + epsilon
        E1 = spin_glass_energy(params_1, J)
        E1.backward()
        I1 = epsilon * params.grad
        params.grad.zero_()

        mask_descent = (I1 < 0).to(device)
        mask_ascent  = (I1 > 0).to(device)
        epsilon_prime = torch.where(mask_descent, alpha * epsilon, epsilon)
        epsilon_prime = (torch.where(mask_ascent, -epsilon, epsilon_prime)).detach().to(device).requires_grad_(False)

        params_2 = params + epsilon_prime
        E2 = spin_glass_energy(params_2, J)
        E2.backward()
        I2 = epsilon_prime * params.grad
        params.grad.zero_()

        # Full CPA update: I2 < 0 or (I1 < 0 and I2 >= 0)
        update_mask = (I2 < 0) | ((I1 < 0) & (I2 >= 0)).to(device)
        perturbation = torch.where((I2 < 0), epsilon_prime, epsilon)
        perturbation = perturbation * update_mask.float()

        with torch.no_grad():
            params += perturbation
            E_final = spin_glass_energy(params, J)
            losses.append(E_final.item() / N)

        ndda_vals.append(compute_NDDA(params, J, alpha))

        # ---- Hessian analysis every 100 epochs ----
        if epoch % 100 == 0:
            mu_Hessian = params.clone().detach().to(device).requires_grad_(True)
            y = spin_glass_energy(mu_Hessian, J)
            grad = torch.autograd.grad(y, mu_Hessian, create_graph=True)[0]

            dim = mu_Hessian.shape[0]
            H = torch.zeros(dim, dim)
            for i in range(dim):
                grad2 = torch.autograd.grad(grad[i], mu_Hessian, create_graph=True)[0]
                H[i] = grad2
            eigenvalues, _ = torch.linalg.eig(H)
            eigenvalues = eigenvalues.real
            min_eigenvalues.append(torch.min(eigenvalues).item())

            # Store additional diagnostics (optional)
            # num_nonnegative = (eigenvalues >= 0).sum().item()
            # total = eigenvalues.numel()
            # etc.
        else:
            # keep last value for continuous time-series
            if len(min_eigenvalues) > 0:
                min_eigenvalues.append(min_eigenvalues[-1])
            else:
                min_eigenvalues.append(None)

        if epoch % 100 == 0:
            print(f"epoch: {epoch:5d}  CPA  loss/N: {losses[-1]:.6f}  NDDA: {ndda_vals[-1]:.4f} min_eigenvalue: {min_eigenvalues[-1]:.6f}")

    return losses, ndda_vals, min_eigenvalues

# ----------------------------------------------------------------------
# GD / PGD optimizer with Hessian analysis
# ----------------------------------------------------------------------
def optimize_GD_or_PGD(params_init, J, sigma_begin=0.02, sigma_end=0.001, method="GD", epochs=10):
    J = J.clone().detach().to(device).requires_grad_(False)
    params = params_init.clone().detach().to(device).requires_grad_(True)
    losses = []
    ndda_vals = []
    min_eigenvalues = []

    for epoch in range(epochs):
        lrlr = sigma_begin * (1. - epoch/epochs) + sigma_end * epoch/epochs
        energy = spin_glass_energy(params, J)
        losses.append(energy.item() / N)
        energy.backward()

        with torch.no_grad():
            if method == "GD":
                params -= lrlr * params.grad
            elif method == "PGD":
                noise = torch.randn_like(params)
                params -= lrlr * (params.grad + noise)

        params.grad.zero_()
        ndda_vals.append(compute_NDDA(params, J))

        # Hessian analysis every 100 epochs
        if epoch % 100 == 0:
            mu_Hessian = params.clone().detach().to(device).requires_grad_(True)
            y = spin_glass_energy(mu_Hessian, J)
            grad = torch.autograd.grad(y, mu_Hessian, create_graph=True)[0]

            dim = mu_Hessian.shape[0]
            H = torch.zeros(dim, dim)
            for i in range(dim):
                grad2 = torch.autograd.grad(grad[i], mu_Hessian, create_graph=True)[0]
                H[i] = grad2
            eigenvalues, _ = torch.linalg.eig(H)
            eigenvalues = eigenvalues.real
            min_eigenvalues.append(torch.min(eigenvalues).item())
        else:
            if len(min_eigenvalues) > 0:
                min_eigenvalues.append(min_eigenvalues[-1])
            else:
                min_eigenvalues.append(None)

        if epoch % 100 == 0:
            print(f"epoch: {epoch:5d}  {method}  loss/N: {losses[-1]:.6f}  NDDA: {ndda_vals[-1]:.4f}  "
                  f"min_eigenvalue: {min_eigenvalues[-1]:.6f}")

    return losses, ndda_vals, min_eigenvalues

# ----------------------------------------------------------------------
# Run experiment
# ----------------------------------------------------------------------
N = 1000
epoch_no = 80000
torch.manual_seed(0)

# J matrix construction (same as original)
J_1 = torch.randn((N//2, N//2), device=device) * 1
J_2 = torch.randn((N//2, N//2), device=device) * 3
J_3 = torch.randn((N//2, N//2), device=device) * 5
J = torch.rand((N, N), device=device)
J[:N//2, :N//2] = J_1 * math.sqrt(1/N)
J[N//2:, :N//2] = J_2 * math.sqrt(1/N)
J[:N//2, N//2:] = J_2 * math.sqrt(1/N)
J[N//2:, N//2:] = J_3 * math.sqrt(1/N)
J = (J + J.T) / 2
J = J * math.sqrt(1/N)
J.diagonal().zero_()

params0 = 0.4 * torch.randn(N, device=device)
logvar0 = -0.1 * torch.ones(N, device=device)   # not used in the new functions, but kept for compatibility

# Run optimizers
gd_losses, gd_ndda, gd_min_eig = optimize_GD_or_PGD(params0, J, sigma_begin=0.2, sigma_end=0.001, method="GD", epochs=epoch_no)

cpa_losses, cpa_ndda, cpa_min_eig = optimize_CPA(params0, J, sigma_begin=0.2, sigma_end=0.001, epochs=epoch_no, alpha=1.3)
pgd_losses, pgd_ndda, pgd_min_eig = optimize_GD_or_PGD(params0, J, sigma_begin=0.2, sigma_end=0.001, method="PGD", epochs=epoch_no)


print("CPA final loss:", cpa_losses[-1])
print("GD final loss:", gd_losses[-1])
print("PGD final loss:", pgd_losses[-1])

# ----------------------------------------------------------------------
# Plots
# ----------------------------------------------------------------------
# Energy
plt.figure(figsize=(9,5))
plt.plot(gd_losses, label="GD")
plt.plot(pgd_losses, label="PGD")
plt.plot(cpa_losses, label="CPA")
plt.xlabel("Epoch")
plt.ylabel("Energy")
plt.legend()
plt.title(f"Spin Glass Variational Optimization (N={N})")
plt.savefig("energy.png")
plt.close()

# NDDA index
plt.figure(figsize=(9,5))
plt.plot(gd_ndda, label="GD")
plt.plot(pgd_ndda, label="PGD")
plt.plot(cpa_ndda, label="CPA")
plt.xlabel("Epoch")
plt.ylabel("NDDA")
plt.legend()
plt.title(f"NDDA Index (N={N})")
plt.savefig("ndda.png")
plt.close()

# Most negative eigenvalue (min eigenvalue)
plt.figure(figsize=(9,5))
# Only plot if values exist (skip Nones)
cpa_min_eig_arr = np.array([v if v is not None else np.nan for v in cpa_min_eig])
pgd_min_eig_arr = np.array([v if v is not None else np.nan for v in pgd_min_eig])
gd_min_eig_arr  = np.array([v if v is not None else np.nan for v in gd_min_eig])
plt.plot(gd_min_eig_arr, label="GD")
plt.plot(pgd_min_eig_arr, label="PGD")
plt.plot(cpa_min_eig_arr, label="CPA")
plt.xlabel("Epoch")
plt.ylabel("Minimum Eigenvalue")
plt.legend()
plt.title(f"Most Negative Eigenvalue (N={N})")
plt.savefig("min_eigenvalue.png")
plt.close()

epoch:     0  GD  loss/N: -0.000049  NDDA: 0.3220  min_eigenvalue: -0.143683
epoch:   100  GD  loss/N: -0.015337  NDDA: 0.1790  min_eigenvalue: -0.092375
epoch:   200  GD  loss/N: -0.028599  NDDA: 0.1140  min_eigenvalue: -0.063260
epoch:   300  GD  loss/N: -0.034712  NDDA: 0.0680  min_eigenvalue: -0.048573
epoch:   400  GD  loss/N: -0.038063  NDDA: 0.0510  min_eigenvalue: -0.034189
epoch:   500  GD  loss/N: -0.039988  NDDA: 0.0420  min_eigenvalue: -0.028262
epoch:   600  GD  loss/N: -0.041171  NDDA: 0.0370  min_eigenvalue: -0.030141
epoch:   700  GD  loss/N: -0.042045  NDDA: 0.0200  min_eigenvalue: -0.033252
epoch:   800  GD  loss/N: -0.042753  NDDA: 0.0200  min_eigenvalue: -0.028901
epoch:   900  GD  loss/N: -0.043287  NDDA: 0.0310  min_eigenvalue: -0.030351
epoch:  1000  GD  loss/N: -0.043699  NDDA: 0.0260  min_eigenvalue: -0.032553
epoch:  1100  GD  loss/N: -0.044042  NDDA: 0.0180  min_eigenvalue: -0.014048
epoch:  1200  GD  loss/N: -0.044318  NDDA: 0.0190  min_eigenvalue: -0.012357